In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

In [3]:
# Tool creation
@tool
def calculator(a : float, b : float, operation : str) -> float:
    """Perform a basic arithmetic operation on two numbers.
    Supported operations are: add, subtract, multiply & divide."""
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            return "Cannot divide by zero"
        return a / b
    else:
        return "Invalid Operation"

In [4]:
calculator.invoke(
    {
        "a" : 45,
        "b" : 55,
        "operation" : "add"
    }
)

100.0

In [5]:
llm = ChatOpenAI(
    model = "openai/gpt-oss-20b",
    api_key = os.getenv("GROK_API_KEY"),
    base_url = "https://api.groq.com/openai/v1",
)

In [6]:
# Tool binding
llm_with_tools = llm.bind_tools(
    [calculator]
)

In [7]:
# Tool calling
response = llm_with_tools.invoke(
    "What is 125 multiplied by 8"
)

In [8]:
response.tool_calls

[{'name': 'calculator',
  'args': {'a': 125, 'b': 8, 'operation': 'multiply'},
  'id': 'fc_56bd9266-3b7b-4db8-bd3c-b3b884bfe91d',
  'type': 'tool_call'}]

In [9]:
# Tool execution
tool_result = calculator.invoke(
    response.tool_calls[0]
)

In [10]:
tool_result

ToolMessage(content='1000.0', name='calculator', tool_call_id='fc_56bd9266-3b7b-4db8-bd3c-b3b884bfe91d')

In [11]:
# Sending the message back to the LLM
messages = [
    HumanMessage("What is 125 multiplied by 8"),
    response,
    tool_result
]
final_result = llm_with_tools.invoke(messages)
print(final_result.content)

1000
